# MultiRate — train all warehouse history and score the latest date

Run All first refreshes FMP equity prices, model fundamentals, and macro data through `quant-warehouse`, then trains the current warehouse equity model through the latest stored equity-price date, including the current partial year. Training and prediction use `quant-orchestrator`; features and labels come from `quant-warehouse`. The model uses the same architecture, supervised heads, and reconstruction objectives as the completed equities-only research run.

The resulting scores describe the last stored date after fitting on all available history; they are not an out-of-sample backtest. Symbols without a price on that common date are reported and excluded from the leaderboard.

Options use a simple rule: expiry nearest **60 calendar days**, then strike nearest the same-date equity close. Long signals select calls; short signals select puts. Expiry-distance ties prefer the later expiry, and strike-distance ties prefer the lower strike. No option model or LLM ranks contracts. Order plans remain optional and require manual review/submission in the existing application.

In [ ]:
# Edit settings here, then Run All.
import os
import sys
from pathlib import Path
from datetime import datetime, timezone

MIN_MARKET_CAP = 10_000_000_000
REFRESH_FMP_DATA = True  # Refresh the selected universe before training.
FMP_HISTORY_START = "1900-01-01"
FMP_REFRESH_STALENESS_DAYS = 60
FMP_REFRESH_ONCE_PER_DATE = True  # Skip FMP datasets already attempted on the current local date.
FMP_REFRESH_MAX_WORKERS = 4
EPOCHS = 1
BATCH_SIZE = 64
D_MODEL = 64
NUM_HEADS = 4
LAYERS = 2
RECONSTRUCTION_WEIGHT = 0.1
REUSE_MODEL_TRAINED_TODAY = True  # Reuse a compatible completed model from the current local date.
CHECKPOINT_EVERY_BATCHES = 10
PROGRESS_UPDATES_PER_EPOCH = 10
SEED = 0
DEVICE = "cuda"
POLARS_THREADS = 8
OMP_THREADS = 4

TOP_K = 20
MIN_LONG_SCORE = 0.50
OPTION_TENOR_DAYS = 60
OPTION_STRATEGY_ALLOCATION = 100_000.0
BUILD_ORDER_PLANS = False  # Enable to read existing Alpaca accounts and prepare reviewable plans.
ALPACA_LIVE_OPTION_DISCOUNT_PCT = float(os.getenv("TRADING_APP_V2_ALPACA_LIVE_OPTION_DISCOUNT_PCT", "90.0"))

os.environ["POLARS_MAX_THREADS"] = str(POLARS_THREADS)
os.environ["OMP_NUM_THREADS"] = str(OMP_THREADS)
roots = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "optimal_trader"]
REPO_ROOT = next(root for root in roots if (root / "app/trading_app_v2_runtime.py").exists())
ORCHESTRATOR_ROOT = Path(os.getenv("QUANT_ORCHESTRATOR_ROOT", str(REPO_ROOT.parent / "quant-orchestrator")))
sys.path.insert(0, str(REPO_ROOT))
if ORCHESTRATOR_ROOT.exists():
    sys.path.insert(0, str(ORCHESTRATOR_ROOT))
UNIVERSE_TAG = f"{MIN_MARKET_CAP / 1e12:g}T" if MIN_MARKET_CAP >= 1e12 else f"{MIN_MARKET_CAP / 1e9:g}B"
RUN_ID = datetime.now(timezone.utc).strftime("latest_%Y%m%dT%H%M%S_%fZ")
RUN_DIR = ORCHESTRATOR_ROOT / "artifacts/multirate_recovery" / UNIVERSE_TAG / RUN_ID
LIVE_DIR = REPO_ROOT / "artifacts/trading_app_v2/multirate_live" / UNIVERSE_TAG / RUN_ID

In [ ]:
import json
import pandas as pd
import torch
from dotenv import load_dotenv
from quant_warehouse.warehouse.api import Warehouse
from quant_orchestrator.research_tools.warehouse_live import train_latest_warehouse_model
from app.trading_app_v2_runtime import (
    alpaca_client_from_env,
    build_alpaca_equity_orders,
    build_latest_equity_leaderboard,
    build_ranked_alpaca_option_orders,
    load_multirate_strategy_scores,
    select_atm_options,
    save_live_artifacts,
    write_streamlit_leaderboard_app,
)

load_dotenv(REPO_ROOT / ".env", override=False)
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("This run requires a CUDA notebook kernel.")
warehouse = Warehouse()
LIVE_DIR.mkdir(parents=True, exist_ok=False)
display({"training_output": str(RUN_DIR), "live_output": str(LIVE_DIR),
         "min_market_cap": MIN_MARKET_CAP, "options_training": False,
         "score_date_policy": "latest finite equity close stored in the selected warehouse universe"})

## Refresh FMP data

Use the same eligibility screening and parallel historical backfill APIs as `trading_app_v2_thetadata_refresh.ipynb` and `trading_app_v2_equity_meta_model.ipynb`. Screening applies `MIN_MARKET_CAP`, excludes unsupported assets, and requires stored warehouse history. The refresh handles missing or stale data and skips fresh sections, using the warehouse historical floor `1900-01-01`. Refresh quarterly and annual fundamentals for this model, prices, and common macro series; save the summary before training. Configure freshness and worker limits in the settings cell. Company news and ThetaData option chains are separate datasets.


In [ ]:
from quant_warehouse.migrate.backfill_missing_fmp import backfill_missing_fmp_historical
from quant_warehouse.platforms.data_providers.fmp.sections import fmp_issuer_model_sections
from quant_warehouse.research_tools.feature_family_eval import FamilyEvaluationConfig, screen_fmp_equity_universe

print(f"FMP refresh kernel: {sys.executable}", flush=True)

refresh_feature_config = FamilyEvaluationConfig(
    provider="fmp", market_cap_min=MIN_MARKET_CAP,
    exchanges=("NASDAQ", "NYSE"), start_date=FMP_HISTORY_START,
)
screened_equity_symbols, raw_fmp_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(
    refresh_feature_config, warehouse=warehouse,
)
print(f"FMP universe source={universe_source}; eligible_symbols={len(screened_equity_symbols):,}; "
      f"min_market_cap={MIN_MARKET_CAP:,}")
display(pd.DataFrame({"symbol": list(screened_equity_symbols)}))
display(universe_eligibility.head(100))


def notebook_refresh_log(message: str) -> None:
    print(f"[fmp-refresh] {message}", flush=True)


# Cover the catalog universe consumed by training, including shared issuer sources.
training_profiles = warehouse.catalog.query_symbol_profiles(
    provider="fmp", min_market_cap=MIN_MARKET_CAP, country="US",
    exchanges=["NASDAQ", "NYSE"], exclude_etf=True, exclude_fund=True,
    supported_equities_only=True,
)
training_refresh_symbols = [profile.symbol for profile in training_profiles
                            if warehouse.read_prices(profile.symbol, provider="fmp").height >= 2]
issuer_sources = {"GOOG": "GOOGL", "BRK-A": "BRK-B"}
refresh_equity_symbols = sorted(set(screened_equity_symbols) | set(training_refresh_symbols)
                               | {issuer_sources.get(symbol, symbol) for symbol in training_refresh_symbols})
print(f"FMP refresh symbols={len(refresh_equity_symbols):,}; training symbols={len(training_refresh_symbols):,}")

fmp_refresh_summary = {
    "status": "skipped_by_config", "equity_symbols": refresh_equity_symbols,
    "etf_symbols": [], "periods": {},
}
fmp_refresh_summary_path = LIVE_DIR / "fmp_refresh_summary.json"
if REFRESH_FMP_DATA:
    for period in ("quarter", "annual"):
        print(f"[fmp-refresh] period={period}", flush=True)
        period_summary = backfill_missing_fmp_historical(
            warehouse=warehouse, equity_provider="fmp", etf_provider="fmp",
            equity_symbols=refresh_equity_symbols, etf_symbols=(), period=period,
            equity_sections=fmp_issuer_model_sections(period),
            include_macro=(period == "quarter"), include_prices=(period == "quarter"),
            macro_start_date=FMP_HISTORY_START,
            staleness_days=FMP_REFRESH_STALENESS_DAYS,
            skip_if_fetched_today=FMP_REFRESH_ONCE_PER_DATE,
            max_workers=FMP_REFRESH_MAX_WORKERS, progress_logger=notebook_refresh_log,
        )
        fmp_refresh_summary["periods"][period] = period_summary
        fmp_refresh_summary["status"] = "in_progress"
        fmp_refresh_summary_path.write_text(json.dumps(fmp_refresh_summary, indent=2, default=str))
        display(pd.DataFrame([{
            "period": period, "equity_symbols": len(period_summary.get("equity_symbols", [])),
            "prices": period_summary.get("equity_prices", {}),
            "fundamentals": period_summary.get("equity", {}),
        }]))
        errors = sum(period_summary.get(key, {}).get("error", 0) for key in ("equity_prices", "equity"))
        macro = period_summary.get("macro")
        if isinstance(macro, list):
            errors += sum(item.get("status") == "error" for item in macro)
        if errors:
            error_details = period_summary.get("equity_errors", [])
            if isinstance(macro, list):
                error_details = error_details + [item for item in macro if item.get("status") == "error"]
            if error_details:
                display(pd.DataFrame(error_details).head(20))
            fmp_refresh_summary["status"] = "error"
            fmp_refresh_summary_path.write_text(json.dumps(fmp_refresh_summary, indent=2, default=str))
            raise RuntimeError(f"FMP refresh reported {errors} errors; inspect {fmp_refresh_summary_path} before training.")
    missing_market_cap = []
    for symbol in training_refresh_symbols:
        issuer = issuer_sources.get(symbol, symbol)
        market = warehouse.read_fundamentals(issuer, provider="fmp", section="historical_market_cap")
        if ("market_cap" not in market.columns or market["market_cap"].drop_nulls().len() == 0):
            missing_market_cap.append({"symbol": symbol, "issuer": issuer, "section": "historical_market_cap"})
    fmp_refresh_summary["required_source_coverage"] = {
        "training_symbols": len(training_refresh_symbols), "missing": missing_market_cap,
    }
    if missing_market_cap:
        fmp_refresh_summary["status"] = "error"
        fmp_refresh_summary_path.write_text(json.dumps(fmp_refresh_summary, indent=2, default=str))
        display(pd.DataFrame(missing_market_cap))
        raise RuntimeError("Required historical market-cap data is missing after FMP refresh; see the coverage table before training.")
    fmp_refresh_summary["status"] = "complete"
else:
    print("FMP refresh disabled; training will use stored warehouse data.")
fmp_refresh_summary_path.write_text(json.dumps(fmp_refresh_summary, indent=2, default=str))
print(f"Wrote FMP refresh summary to {fmp_refresh_summary_path}")


## Train and predict

This calls the shared optimized warehouse workflow: full history from `1900-01-01`, four preparation workers, batched annual memory, all equity supervised objectives, masked reconstruction and next-token prediction. It reuses a compatible completed checkpoint from the current local date when available; otherwise it starts fresh weights, saves a checkpoint, and scores only the latest warehouse date using that year's available context. The preceding cell refreshes FMP data before the workflow determines its latest scoring date.

In [ ]:
training_result = train_latest_warehouse_model(
    RUN_DIR, min_market_cap=MIN_MARKET_CAP, epochs=EPOCHS, batch_size=BATCH_SIZE,
    d_model=D_MODEL, num_heads=NUM_HEADS, layers=LAYERS, device=DEVICE, seed=SEED,
    reconstruction_weight=RECONSTRUCTION_WEIGHT,
    reuse_if_trained_today=REUSE_MODEL_TRAINED_TODAY,
    checkpoint_every_batches=CHECKPOINT_EVERY_BATCHES,
    progress_updates_per_epoch=PROGRESS_UPDATES_PER_EPOCH,
    warehouse=warehouse,
)
score_date = training_result["score_date"]
CHECKPOINT_PATH = Path(training_result["checkpoint"])
display(training_result)

In [ ]:
strategy_scores = load_multirate_strategy_scores(Path(training_result["prediction_path"]))
latest_prices = pd.read_parquet(training_result["prices_path"])
assert set(pd.to_datetime(strategy_scores["date"]).dt.strftime("%Y-%m-%d")) == {score_date}
price_map = latest_prices.set_index("symbol")["close"].to_dict()
leaderboard = build_latest_equity_leaderboard(
    strategy_scores, top_k=TOP_K, min_long_score=MIN_LONG_SCORE, price_map=price_map,
)
display(leaderboard.head(TOP_K + 5))
print({"score_date": score_date, "scored_equities": len(strategy_scores),
       "missing_latest_prices": training_result["missing_latest_prices"]})

## Select live Alpaca options near 60 DTE

Use current Alpaca live option contracts only. Selection is deterministic: choose the expiration nearest 60 DTE, then the strike nearest the saved equity price for the model direction. ThetaData is not used for live option selection.


In [ ]:
alpaca_option_data_client = alpaca_client_from_env("OPTION", live=True)
option_rankings, option_selection_audit = select_atm_options(
    leaderboard, score_date=score_date, target_dte=OPTION_TENOR_DAYS,
    top_k=TOP_K, alpaca_client=alpaca_option_data_client,
)
selected_symbols = option_rankings["symbol"].tolist()
option_leaderboard = leaderboard.loc[leaderboard["symbol"].isin(selected_symbols)].copy()
option_leaderboard["selected"] = True
option_selection_audit.to_csv(LIVE_DIR / "option_selection_audit.csv", index=False)
display(option_rankings)
display(option_selection_audit)

In [ ]:
paper_order_plans = {}
if BUILD_ORDER_PLANS:
    paper_order_plans["alpaca_equity_paper"] = build_alpaca_equity_orders(
        leaderboard=leaderboard, account_prefix="EQUITY", gross_exposure=0.95,
    )
    for name, live in [("alpaca_option_paper", False), ("alpaca_option_live", True)]:
        paper_order_plans[name] = build_ranked_alpaca_option_orders(
            option_rankings=option_rankings, decisions=option_leaderboard[["symbol", "direction"]],
            account_prefix="OPTION", strategy_allocation=OPTION_STRATEGY_ALLOCATION,
            max_underlyings=TOP_K, live=live,
            discount_pct=ALPACA_LIVE_OPTION_DISCOUNT_PCT if live else 0.0,
        )
    for name, frame in paper_order_plans.items():
        print(name, len(frame))
        display(frame)
else:
    print("Predictions and option selections are ready. Order-plan generation is disabled.")

In [ ]:
saved = save_live_artifacts(
    live_dir=LIVE_DIR, leaderboard=leaderboard, symbol_scores=strategy_scores,
    option_ml_rankings=option_rankings, orders=paper_order_plans,
)
streamlit_app = write_streamlit_leaderboard_app(
    live_dir=LIVE_DIR, leaderboard=leaderboard, symbol_scores=strategy_scores,
    option_ml_rankings=option_rankings, orders=paper_order_plans,
)
(LIVE_DIR / "model_run.json").write_text(json.dumps(training_result, indent=2))
print({"checkpoint": str(CHECKPOINT_PATH), "score_date": score_date,
       "saved": saved, "streamlit_app": str(streamlit_app)})
import socket
import subprocess
import time
from urllib.request import urlopen
from IPython.display import HTML

existing_server = globals().get("_streamlit_process")
if (existing_server is not None and existing_server.poll() is None
        and globals().get("_streamlit_app_path") == str(streamlit_app)):
    STREAMLIT_URL = globals()["_streamlit_url"]
else:
    with socket.socket() as probe:
        probe.bind(("127.0.0.1", 0))
        streamlit_port = probe.getsockname()[1]
    STREAMLIT_URL = f"http://127.0.0.1:{streamlit_port}"
    streamlit_log_path = LIVE_DIR / "streamlit.log"
    with streamlit_log_path.open("ab") as streamlit_log:
        _streamlit_process = subprocess.Popen(
            [sys.executable, "-m", "streamlit", "run", str(streamlit_app),
             "--server.headless", "true", "--server.address", "127.0.0.1",
             "--server.port", str(streamlit_port), "--browser.gatherUsageStats", "false"],
            stdout=streamlit_log, stderr=subprocess.STDOUT, start_new_session=True,
        )
    _streamlit_app_path = str(streamlit_app)
    _streamlit_url = STREAMLIT_URL
    for _ in range(60):
        if _streamlit_process.poll() is not None:
            tail = streamlit_log_path.read_text(errors="replace")[-4000:]
            raise RuntimeError(f"Streamlit exited during startup. Log: {streamlit_log_path}\n{tail}")
        try:
            with urlopen(f"{STREAMLIT_URL}/_stcore/health", timeout=1) as response:
                if response.status == 200:
                    break
        except OSError:
            time.sleep(0.25)
    else:
        raise RuntimeError(f"Streamlit did not become ready; inspect {streamlit_log_path}")

print(f"Streamlit server: {STREAMLIT_URL}")
display(HTML(f'<a href="{STREAMLIT_URL}" target="_blank">Open Streamlit app</a>'))